### 資料清理

In [1]:
import pandas as pd
import numpy as np

user = pd.read_csv("../data/user_data.csv")

review = pd.read_csv(
    "../data/dp001_review.csv",
    engine="python",
    on_bad_lines="skip"
)

prac = pd.read_csv("../data/dp001_prac.csv")

In [30]:
print(user.info())
print(review.info())
print(prac.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1081 entries, 0 to 1080
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_sn          1081 non-null   int64  
 1   organization_id  1081 non-null   object 
 2   grade            1081 non-null   int64  
 3   class            1081 non-null   int64  
 4   chinese_score    1065 non-null   float64
 5   math_score       1074 non-null   float64
 6   english_score    1057 non-null   float64
dtypes: float64(3), int64(3), object(1)
memory usage: 59.2+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16257 entries, 0 to 16256
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_sn          16257 non-null  int64  
 1   review_sn        16257 non-null  int64  
 2   indicator_name   16257 non-null  object 
 3   subject_name     16257 non-null  object 
 4   video_name       16257

In [31]:
user["user_sn"] = user["user_sn"].astype(str)
review["user_sn"] = review["user_sn"].astype(str)
prac["user_sn"] = prac["user_sn"].astype(str)

In [32]:
# dp001_review
review["start_time"] = pd.to_datetime(review["start_time"], errors="coerce")
review["end_time"] = pd.to_datetime(review["end_time"], errors="coerce")

# dp001_prac
prac["date"] = pd.to_datetime(prac["date"], errors="coerce")

In [33]:
user = user.dropna(subset=[
    "grade",
    "chinese_score",
    "math_score",
    "english_score"
])

review = review.dropna(subset=[
    "user_sn",
    "video_len",
    "finish_rate",
    "start_time",
    "end_time"
])

review = review[
    (review["video_len"] > 0) &
    (review["finish_rate"].between(0, 100))
]

prac = prac.dropna(subset=[
    "user_sn",
    "score_rate"
])

prac["score_rate"] = pd.to_numeric(prac["score_rate"], errors="coerce")

prac.loc[~prac["score_rate"].between(0, 100), "score_rate"] = np.nan

prac["score_rate"] = prac["score_rate"] / 100

In [34]:
print("User:", user.shape)
print("Review:", review.shape)
print("Prac:", prac.shape)

print("Review missing rate:")
print(review.isna().mean())

print("Prac missing rate:")
print(prac.isna().mean())

User: (1045, 7)
Review: (15488, 11)
Prac: (25060, 9)
Review missing rate:
user_sn            0.0
review_sn          0.0
indicator_name     0.0
subject_name       0.0
video_name         0.0
video_len          0.0
start_timestamp    0.0
start_time         0.0
end_timestamp      0.0
end_time           0.0
finish_rate        0.0
dtype: float64
Prac missing rate:
prac_sn           0.0
user_sn           0.0
date              0.0
during_time       0.0
score_rate        0.0
binary_res        0.0
items_ans_time    0.0
indicator_name    0.0
subject_name      0.0
dtype: float64


In [35]:
user.to_csv("../outputs/user_clean.csv", index=False)
review.to_csv("../outputs/dp001_review_clean.csv", index=False)
prac.to_csv("../outputs/dp001_prac_clean.csv", index=False)